# 11 — A Real Multi-Agent System

Module 10 taught the protocol with agents that reversed strings and counted to
ten. This is the same protocol with agents that do something.

```
                    supervisor  (9200)
                         │
         ┌───────────────┼───────────────┐
         ▼               ▼               ▼
     vdb (9201)     cypher (9202)    chart (9203)
     Pinecone         Neo4j          Plotly spec
     module 05        module 08
```

Nothing new is built here. Modules 05 and 08 already made the capabilities; this
wraps each one in an A2A agent so a supervisor can use them without importing
them.

That is the point. **The agents share no code at runtime.** They could be in four
repositories, four languages, four accounts. What holds them together is a
contract in the message.

| § | What |
|---|---|
| 0 | Setup, and choosing a provider |
| 1 | The contract — the most important file |
| 2 | The three specialists |
| 3 | The supervisor's two kinds of decision |
| 4 | A whole turn |
| 5 | What breaks, and what happens then |

---
## 0. Setup

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import os

# Either provider works. The architecture does not care, which is why both code
# paths exist and one variable picks between them.
os.environ.setdefault("PROVIDER", "openai")        # or "anthropic"
os.environ.setdefault("OPENAI_API_KEY", "")
os.environ.setdefault("ANTHROPIC_API_KEY", "")

# The stores from modules 05 and 08.
os.environ.setdefault("PINECONE_API_KEY", "")
os.environ.setdefault("INDEX_NAME", "rag-docs")
os.environ.setdefault("NEO4J_URI", "bolt://localhost:7687")
os.environ.setdefault("NEO4J_USER", "neo4j")
os.environ.setdefault("NEO4J_PASSWORD", "")

In [ ]:
import asyncio
import json
import sys
from pathlib import Path

from aiohttp import web

HERE = Path.cwd()
sys.path.insert(0, str(HERE))
COURSE = HERE.parent.parent
sys.path.insert(0, str(COURSE / "10-a2a-protocol"))   # the protocol library
sys.path.insert(0, str(COURSE / "05-rag"))            # the rag package

from a2a_mini import A2AClient
from agents_lib import config, contracts, llm

print(f"provider  {llm.describe()}")
print(f"ports     {config.PORTS}")

---
## 1. The contract

`agents_lib/contracts.py` is the shortest file in the module and the most
important one.

Here is the problem it solves. The supervisor cannot see inside a specialist —
not its memory, not its prompt, not what it did. All it receives is an artifact.

So "here are some results" is not enough. Are they rows it could chart? Passages
it should quote? Nothing at all? Guessing produces a system that works on the
questions you happened to try.

Every specialist therefore **declares its shape**, and the supervisor routes on
that.

In [ ]:
# Every specialist returns one of these.
print(json.dumps(contracts.table(["sponsor", "trials"],
                                 [["Gilead", 4], ["Novartis", 2]]), indent=2))
print()
print(json.dumps(contracts.empty("no trials match that sponsor"), indent=2))

`empty` and `error` are separate on purpose.

*"No trials match that sponsor"* is an **answer** — the analyst learns something
true. *"Neo4j refused the connection"* is a **failure**.

Collapse them into one shape and the supervisor tells someone there is no data
when the database was actually down. That is the worse of the two outcomes, and
it is the one you get by default.

In [ ]:
# What the supervisor's MODEL sees of a 43-row result.
big = contracts.table(["sponsor", "trials"],
                      [[f"Sponsor {i}", i % 5 + 1] for i in range(43)])

print("full result :", len(json.dumps(big)), "chars")
print("model sees  :", contracts.summarise_for_model(big))

The model routes and writes prose. It does not process data.

Hand it 43 rows and it will sometimes retype values into its answer — and a
retyped NCT number that is one digit wrong reads exactly like a correct one.
There is no error, no warning, and someone acts on it.

The full result stays in the supervisor and travels to the chart agent by
reference. The model gets shape, size and three examples.

---
## 2. The three specialists

Each is a thin wrapper. The capability already existed.

In [ ]:
import agents.vdb_agent as vdb_agent
import agents.cypher_agent as cypher_agent
import agents.chart_agent as chart_agent
import agents.supervisor as supervisor_agent

for module in (vdb_agent, cypher_agent, chart_agent, supervisor_agent):
    card = module.CARD
    streams = "streams" if card.capabilities["streaming"] else "no streaming"
    print(f"{card.name:<11}{streams:<14}{card.skills[0].name}")

Read the two data agents' skill descriptions carefully — they are what the
supervisor's model routes on.

In [ ]:
for module in (vdb_agent, cypher_agent):
    print(f"── {module.CARD.name}")
    print(f"   {module.CARD.skills[0].description}\n")

Notice they say what they are **not** for. `vdb` says *"not for counting,
comparing across trials, or relationships"*; `cypher` says *"not for what a
protocol document says"*.

Without that, both descriptions win every question. The router picks one
arbitrarily and looks like it is working until a question lands on the wrong
side.

### The Cypher guard

A model writes the query. That is the point, and also the risk — so the query is
checked in code before it runs.

This is the split worth internalising: **the prompt can be edited, tuned,
replaced. The guard cannot.** Anything that must be true regardless of what the
model produces belongs in code, where it is reviewed and shows up in a diff.

In [ ]:
queries = [
    "MATCH (s:Sponsor)<-[:SPONSORED_BY]-(t:Trial) RETURN s.name, count(t) LIMIT 100",
    "MATCH (n) DETACH DELETE n",
    "MATCH (t:Trial) SET t.phase = 'X' RETURN t",
    "CALL apoc.periodic.iterate('MATCH (n) RETURN n','DELETE n',{})",
    "MATCH (t:Trial) RETURN t.deleted_at LIMIT 10",
    "MATCH (s:Sponsor) WHERE s.name = 'Reset Pharma' RETURN s",
]
for q in queries:
    ok, why = cypher_agent._is_read_only(q)
    print(f"{'allow ' if ok else 'REFUSE':<8}{q[:62]}")
    if not ok:
        print(f"        {why}")

The last two matter as much as the refusals. `deleted_at` is a property name and
`Reset Pharma` is a sponsor; neither is a write.

A guard that fires on legitimate queries gets switched off, and then it protects
nothing. Matching whole words rather than substrings is the difference.

### The chart agent sanitises its own output

Plotly renders a limited set of HTML in titles, axis labels and hover text. So a
string in the figure spec reaches a browser's DOM.

The values being charted come from **the data**, not the model: sponsor names,
site names, drug names. The realistic path is not the model misbehaving — it is
the model faithfully copying whatever a name field contains into a chart label.

In [ ]:
labels = [
    "Trials by Sponsor",
    "Enrolment<br>by Country",
    "<b>Phase 3</b> only",
    "Dose <sub>max</sub> mg",
    "<img src=x onerror=alert(1)>Acme Pharma",
    "<script>fetch('/steal')</script>Sponsor",
]
for label in labels:
    print(f"{label[:44]:<46} -> {chart_agent._clean(label)}")

Legitimate Plotly markup survives. `<br>` and `<b>` are how analysts get readable
multi-line titles, and stripping everything would break real charts — a guard
that breaks real work gets removed.

The dangerous elements lose their **contents** too, not just their tags. Strip
only the tags from `<script>alert(1)</script>` and `alert(1)` sits in the chart
title as attacker-chosen text displayed as data.

---
## 3. The supervisor's two kinds of decision

**Routing is a judgement.** Nothing but a model reads "which sponsors run more
than one trial" as a counting question. So the model decides.

**Rendering is not.** A table-shaped result should be charted. That is an `if`,
and it is written as one — deliberately not offered to the model as a tool.

A model that forgets a `render` tool ships a chartless answer and nothing errors.
You find out when someone asks where the chart went. So remove the affordance
rather than add prompt language telling it not to forget.

In [ ]:
import inspect

source = inspect.getsource(supervisor_agent.execute)
start = source.index("# ── decision 2")
print(source[start:source.index("# ── compose")])

---
## 4. A whole turn

Start all four in this notebook. Normally each is `python agents/<name>.py` in
its own terminal — and in module 12, four separate deployments.

In [ ]:
runners = []

async def serve(server, port):
    runner = web.AppRunner(server.app())
    await runner.setup()
    await web.TCPSite(runner, "127.0.0.1", port).start()
    runners.append(runner)

for module, name in ((vdb_agent, "vdb"), (cypher_agent, "cypher"),
                     (chart_agent, "chart"), (supervisor_agent, "supervisor")):
    await serve(module.server, config.PORTS[name])

print("four agents listening")

In [ ]:
client = A2AClient(config.endpoint("supervisor"))
await client.discover()


async def ask(question: str):
    """Send a question and print the turn as it unfolds."""
    print(f"Q  {question}\n")
    artifacts = {}
    async for event in client.stream(question):
        if event.get("kind") == "progress":
            print(f"   · {event['note'][:100]}")
        elif event.get("kind") == "artifact-update":
            a = event["artifact"]
            artifacts[a["name"]] = a["parts"][0]["text"]
        if event.get("final"):
            break

    print(f"\n{artifacts.get('answer', '(no answer)')}")
    if "chart" in artifacts:
        chart = json.loads(artifacts["chart"])
        print(f"\nchart: {chart['chart_type']} — {chart['insight']}")
    return artifacts


answer = await ask("which sponsors run more than one trial?")

Read the progress lines. Every one is a hop:

```
found 3 specialists          the supervisor fetched three cards
routing to cypher            the model chose
[cypher] writing Cypher      the specialist's own event, forwarded
[cypher] MATCH (s:Sponsor)…  the query it wrote
[cypher] table: 43 rows      the shape it declared
charting                     code saw shape=table and called chart
```

The `[cypher]` lines came from a different process. The supervisor forwards its
specialists' events upward, so the fan-out is visible instead of a silent pause.

In [ ]:
answer = await ask("what are the exclusion criteria in the protocols?")

Different route, and **no chart**. The result was `passages`, and code only
charts `table`.

The model was never asked. It could not have got this wrong.

---
## 5. What breaks

Four processes, so four things can be down. Here is what each looks like.

In [ ]:
# Stop the graph agent and ask a graph question.
graph_runner = runners[1]
await graph_runner.cleanup()
print("cypher agent stopped\n")

await ask("which sponsors run more than one trial?")

The supervisor discovered two agents instead of three, and the router was only
offered what actually exists. It cannot route to something that is not there.

That is why `_fleet()` runs per request rather than at startup: an agent that was
down when the supervisor booted should not be permanently invisible, and in a
real deployment one specialist restarting is normal.

---
## What this is missing

**Auth.** Nothing here checks anything. Module 12 adds SigV4 — which
authenticates an **IAM principal**, not a person. Worth being clear about the
consequence now: the supervisor will call specialists as itself, so there is no
per-user authorisation downstream. That is fine when every analyst sees the same
data, and it is a real limit if they should not.

**Durability.** Restart the supervisor and every running task is gone. The queue
is a Python list.

**Retries.** A specialist that times out fails the turn. Real deployments retry
transient network faults but not agent errors — retrying a refusal just burns the
budget to be refused again.

**Budget enforcement.** `MAX_AGENT_CALLS` is in the config and this supervisor
makes one call per turn, so nothing enforces it yet. A plan-and-replan loop
would need it, and it belongs in the tool rather than the prompt: a prompt asking
for efficiency is a suggestion, a counter that refuses the fifth call is a wall.

## What to take from this

**The contract is the system.** Four programs that share no code agree on one
thing — a `shape` field — and that is enough to build on.

**Split the decisions.** Judgement to the model, mechanism to code. Routing is
judgement. Charting a table is not.

**Keep bulk data away from the model.** It routes and writes prose. Data travels
between agents, not through it.

In [ ]:
for runner in runners:
    try:
        await runner.cleanup()
    except Exception:
        pass
print("stopped")